# Model evaluation with AdaPT on MNIST dataset

In this notebook you can evaluate different approximate multipliers on various models based on MNIST dataset

Steps:
* Select number of threads to use
* Load dataset
* Load Adapt Layers
* Define Model
* Run model calibration for quantization
* Evaluate


**Note**:
* This notebook should be run on a X86 machine

* Please make sure you have run the installation steps first

In [1]:
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.utils.data.dataloader as dataloader
import torch.optim as optim

from torch.utils.data import TensorDataset
from torch.autograd import Variable
from torchvision import transforms
from torchvision.datasets import MNIST
import tqdm

## Select number of threads to use

For optimal performance set them as the number of your cpu threads (not cpu cores)

In [2]:
threads = 4
torch.set_num_threads(threads)

#maybe better performance
%env OMP_PLACES=cores
%env OMP_PROC_BIND=close
%env OMP_WAIT_POLICY=active

env: OMP_PLACES=cores
env: OMP_PROC_BIND=close
env: OMP_WAIT_POLICY=active


## Load Dataset


In [3]:
train = MNIST('./datasets/mnist_data/data', train=True, download=True, transform=transforms.Compose([
    transforms.ToTensor(), # ToTensor does min-max normalization. 
]), )

test = MNIST('./datasets/mnist_data/data', train=False, download=True, transform=transforms.Compose([
    transforms.ToTensor(), # ToTensor does min-max normalization. 
]), )

# Create DataLoader
# dataloader_args = dict(shuffle=True, batch_size=64,num_workers=1, pin_memory=False)
# train_loader = dataloader.DataLoader(train, **dataloader_args)
# test_loader = dataloader.DataLoader(test, **dataloader_args)

# shuffle=True,
dataloader_args = dict(
    shuffle=False,
    batch_size=64,
    num_workers=0,
    pin_memory=False
)

# train_loader = dataloader.DataLoader(train, **dataloader_args)
# test_loader = dataloader.DataLoader(test, **dataloader_args)

train_loader = dataloader.DataLoader(
    train,
    batch_size=64,
    shuffle=True,
    num_workers=0,
    pin_memory=False
)

test_loader = dataloader.DataLoader(
    test,
    batch_size=64,
    shuffle=False,
    num_workers=0,
    pin_memory=False
)

## Load Adapt Layers

In [4]:
#Load ADAPT layers
from adapt.approx_layers import axx_layers as approxNN

## Choose approximate multiplier 

Two approximate multipliers are already provided

**mul8s_acc** - (header file: mul8s_acc.h)   <--  default

**mul8s_1L2H** - (header file: mul8s_1L2H.h)



In order to use your custom multiplier you need to use the provided tool (LUT_generator) to easily create the C++ header for your multiplier. Then you just place it inside the adapt/cpu-kernels/axx_mults folder. The name of the axx_mult here must match the name of the header file. The same axx_mult is used in all layers. 

Tip: If you want explicitly to set for each layer a different axx_mult you must do it from the model definition using the respective AdaPT_Conv2d class of each layer.

In [7]:
axx_mult = 'vakili_wrapper_generated_clean'

## Define Model

Jit compilation method loads 'on the fly' the C++ extentions of the approximate multipliers. Then the pytorch model is loaded

In [8]:
#set flag for use of AdaPT custom layers or vanilla PyTorch
use_adapt = True
# use_adapt = False

class Model(nn.Module):
    def __init__(self):
        super(Model, self).__init__()

        # FPGA-friendly MLP:
        # 784 -> 64 -> 32 -> 10

        if use_adapt:
            self.fc1 = approxNN.AdaPT_Linear(784, 64, axx_mult=axx_mult)
        else:
            self.fc1 = nn.Linear(784, 64)

        self.bc1 = nn.BatchNorm1d(64)

        if use_adapt:
            self.fc2 = approxNN.AdaPT_Linear(64, 32, axx_mult=axx_mult)
        else:
            self.fc2 = nn.Linear(64, 32)

        self.bc2 = nn.BatchNorm1d(32)

        if use_adapt:
            self.fc3 = approxNN.AdaPT_Linear(32, 10, axx_mult=axx_mult)
        else:
            self.fc3 = nn.Linear(32, 10)

    def forward(self, x):
        x = x.view((-1, 784))

        h = self.fc1(x)
        h = self.bc1(h)
        h = F.relu(h)
        h = F.dropout(h, p=0.2, training=self.training)

        h = self.fc2(h)
        h = self.bc2(h)
        h = F.relu(h)
        h = F.dropout(h, p=0.1, training=self.training)

        h = self.fc3(h)
        out = F.log_softmax(h, -1)
        return out

model = Model()
model.cpu()

Using /root/.cache/torch_extensions as PyTorch extensions root...
No modifications detected for re-loaded extension module PyInit_linear_vakili_wrapper_generated_clean, skipping build step...
Loading extension module PyInit_linear_vakili_wrapper_generated_clean...
Using /root/.cache/torch_extensions as PyTorch extensions root...
No modifications detected for re-loaded extension module PyInit_linear_vakili_wrapper_generated_clean, skipping build step...
Loading extension module PyInit_linear_vakili_wrapper_generated_clean...
Using /root/.cache/torch_extensions as PyTorch extensions root...
No modifications detected for re-loaded extension module PyInit_linear_vakili_wrapper_generated_clean, skipping build step...
Loading extension module PyInit_linear_vakili_wrapper_generated_clean...


Model(
  (fc1): AdaPT_Linear(
    (quantizer): TensorQuantizer(8bit per-tensor amax=dynamic calibrator=HistogramCalibrator quant)
    (quantizer_w): TensorQuantizer(8bit per-tensor amax=dynamic calibrator=HistogramCalibrator quant)
  )
  (bc1): BatchNorm1d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (fc2): AdaPT_Linear(
    (quantizer): TensorQuantizer(8bit per-tensor amax=dynamic calibrator=HistogramCalibrator quant)
    (quantizer_w): TensorQuantizer(8bit per-tensor amax=dynamic calibrator=HistogramCalibrator quant)
  )
  (bc2): BatchNorm1d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (fc3): AdaPT_Linear(
    (quantizer): TensorQuantizer(8bit per-tensor amax=dynamic calibrator=HistogramCalibrator quant)
    (quantizer_w): TensorQuantizer(8bit per-tensor amax=dynamic calibrator=HistogramCalibrator quant)
  )
)

In [9]:
## Train Model

# optimizer = optim.Adam(model.parameters(), lr=0.001)
# criterion = nn.NLLLoss()

# model.train()

# num_epochs = 5

# for epoch in range(num_epochs):
#     running_loss = 0.0
#     correct = 0
#     total = 0

#     for data, target in tqdm.tqdm(train_loader):
#         data = data.cpu()
#         target = target.cpu()

#         optimizer.zero_grad()

#         output = model(data)
#         loss = criterion(output, target)

#         loss.backward()
#         optimizer.step()

#         running_loss += loss.item()

#         pred = output.data.max(1)[1]
#         correct += pred.eq(target.data).cpu().sum().item()
#         total += target.size(0)

#     train_acc = correct / total
#     avg_loss = running_loss / len(train_loader)

#     print(f"Epoch {epoch+1}/{num_epochs}, Loss: {avg_loss:.4f}, Train Accuracy: {train_acc:.4f}")

# torch.save(model.state_dict(), 'models/state_dicts/mnist_fpga_mlp_64_32.pt')

In [10]:
model.load_state_dict(torch.load('models/state_dicts/mnist_fpga_mlp_64_32.pt'))

<All keys matched successfully>

## Run model calibration for quantization

Calibrates the quantization parameters 

Need to re-run it each time the model changes

In [11]:
from pytorch_quantization import nn as quant_nn
from pytorch_quantization import calib

def collect_stats(model, data_loader, num_batches):
     """Feed data to the network and collect statistic"""

     # Enable calibrators
     for name, module in model.named_modules():
         if isinstance(module, quant_nn.TensorQuantizer):
             if module._calibrator is not None:
                 module.disable_quant()
                 module.enable_calib()
             else:
                 module.disable()
        
     evaluate_x = Variable(data_loader.dataset.data.type_as(torch.FloatTensor())).cpu()
     model(evaluate_x)
        
     # Disable calibrators
     for name, module in model.named_modules():
         if isinstance(module, quant_nn.TensorQuantizer):
             if module._calibrator is not None:
                 module.enable_quant()
                 module.disable_calib()
             else:
                 module.enable()

def compute_amax(model, **kwargs):
 # Load calib result
 for name, module in model.named_modules():
     if isinstance(module, quant_nn.TensorQuantizer):
         if module._calibrator is not None:
             if isinstance(module._calibrator, calib.MaxCalibrator):
                 module.load_calib_amax()
             else:
                 module.load_calib_amax(**kwargs)
         print(F"{name:40}: {module}")
 model.cpu()

# It is a bit slow since we collect histograms on CPU
with torch.no_grad():
    stats = collect_stats(model, test_loader, num_batches=2)
    amax = compute_amax(model, method="percentile", percentile=99.99)
    
    # optional - test different calibration methods
    #amax = compute_amax(model, method="mse")
    #amax = compute_amax(model, method="entropy")
    

W0710 19:33:19.942693 128251914852160 tensor_quantizer.py:173] Disable HistogramCalibrator
W0710 19:33:19.944340 128251914852160 tensor_quantizer.py:173] Disable HistogramCalibrator
W0710 19:33:19.945317 128251914852160 tensor_quantizer.py:173] Disable HistogramCalibrator
W0710 19:33:19.946124 128251914852160 tensor_quantizer.py:173] Disable HistogramCalibrator
W0710 19:33:19.947574 128251914852160 tensor_quantizer.py:173] Disable HistogramCalibrator
W0710 19:33:19.948633 128251914852160 tensor_quantizer.py:173] Disable HistogramCalibrator
W0710 19:33:19.953087 128251914852160 tensor_quantizer.py:237] Load calibrated amax, shape=torch.Size([]).
W0710 19:33:19.954161 128251914852160 tensor_quantizer.py:238] Call .cuda() if running on GPU after loading calibrated amax.
W0710 19:33:19.955328 128251914852160 tensor_quantizer.py:237] Load calibrated amax, shape=torch.Size([]).
W0710 19:33:19.958439 128251914852160 tensor_quantizer.py:237] Load calibrated amax, shape=torch.Size([]).
W0710 19

fc1.quantizer                           : TensorQuantizer(8bit per-tensor amax=254.8755 calibrator=HistogramCalibrator quant)
fc1.quantizer_w                         : TensorQuantizer(8bit per-tensor amax=0.3770 calibrator=HistogramCalibrator quant)
fc2.quantizer                           : TensorQuantizer(8bit per-tensor amax=3.5321 calibrator=HistogramCalibrator quant)
fc2.quantizer_w                         : TensorQuantizer(8bit per-tensor amax=0.3853 calibrator=HistogramCalibrator quant)
fc3.quantizer                           : TensorQuantizer(8bit per-tensor amax=7.5230 calibrator=HistogramCalibrator quant)
fc3.quantizer_w                         : TensorQuantizer(8bit per-tensor amax=0.7954 calibrator=HistogramCalibrator quant)


## Evaluate

In [12]:
evaluate_x = Variable(test_loader.dataset.data.type_as(torch.FloatTensor())).cpu()
evaluate_y = Variable(test_loader.dataset.targets).cpu()


output = model(evaluate_x)
pred = output.data.max(1)[1]
d = pred.eq(evaluate_y.data).cpu()
accuracy = d.sum()/d.size()[0]

print('Accuracy:', accuracy)

Accuracy: tensor(0.9564)


In [17]:
# def make_adapt_model(mult_name, weight_path):
#     global axx_mult

#     axx_mult = mult_name

#     model = Model()
#     model.cpu()

#     model.load_state_dict(torch.load(weight_path, map_location='cpu'))
#     model.eval()

#     return model

In [18]:
def collect_stats_loader(model, data_loader, num_batches=10):
    model.eval()

    # Enable calibrators
    for name, module in model.named_modules():
        if isinstance(module, quant_nn.TensorQuantizer):
            if module._calibrator is not None:
                module.disable_quant()
                module.enable_calib()
            else:
                module.disable()

    # Feed correctly transformed batches through the model
    with torch.no_grad():
        for i, (data, target) in enumerate(data_loader):
            if i >= num_batches:
                break

            data = data.cpu()
            model(data)

    # Disable calibrators and enable quantization
    for name, module in model.named_modules():
        if isinstance(module, quant_nn.TensorQuantizer):
            if module._calibrator is not None:
                module.enable_quant()
                module.disable_calib()
            else:
                module.enable()

In [19]:
def make_calibrated_adapt_model(mult_name, weight_path, calib_loader):
    global axx_mult

    axx_mult = mult_name

    model = Model()
    model.cpu()

    model.load_state_dict(torch.load(weight_path, map_location='cpu'))
    model.eval()

    collect_stats_loader(model, calib_loader, num_batches=10)
    compute_amax(model, method="percentile", percentile=99.99)

    model.eval()
    return model

In [21]:
weight_path = 'models/state_dicts/mnist_fpga_mlp_64_32.pt'

model_exact = make_calibrated_adapt_model(
    'mul8s_acc',
    weight_path,
    test_loader
)

model_approx = make_calibrated_adapt_model(
    'vakili_wrapper_generated_clean',
    weight_path,
    test_loader
)
# # or:
# # model_approx = make_adapt_model('mul8s_1L2H', weight_path)

# # Calibrate exact model
# stats = collect_stats(model_exact, test_loader, num_batches=2)
# amax = compute_amax(model_exact, method="percentile", percentile=99.99)

# # Calibrate approximate/dummy model
# stats = collect_stats(model_approx, test_loader, num_batches=2)
# amax = compute_amax(model_approx, method="percentile", percentile=99.99)

W0710 19:35:12.638124 128251914852160 tensor_quantizer.py:173] Disable HistogramCalibrator
W0710 19:35:12.640563 128251914852160 tensor_quantizer.py:173] Disable HistogramCalibrator
W0710 19:35:12.641796 128251914852160 tensor_quantizer.py:173] Disable HistogramCalibrator
W0710 19:35:12.643014 128251914852160 tensor_quantizer.py:173] Disable HistogramCalibrator
W0710 19:35:12.644320 128251914852160 tensor_quantizer.py:173] Disable HistogramCalibrator
W0710 19:35:12.645754 128251914852160 tensor_quantizer.py:173] Disable HistogramCalibrator
W0710 19:35:12.647495 128251914852160 tensor_quantizer.py:237] Load calibrated amax, shape=torch.Size([]).
W0710 19:35:12.649225 128251914852160 tensor_quantizer.py:237] Load calibrated amax, shape=torch.Size([]).
W0710 19:35:12.651326 128251914852160 tensor_quantizer.py:237] Load calibrated amax, shape=torch.Size([]).
W0710 19:35:12.653043 128251914852160 tensor_quantizer.py:237] Load calibrated amax, shape=torch.Size([]).
W0710 19:35:12.654356 1282

Using /root/.cache/torch_extensions as PyTorch extensions root...
No modifications detected for re-loaded extension module PyInit_linear_mul8s_acc, skipping build step...
Loading extension module PyInit_linear_mul8s_acc...
Using /root/.cache/torch_extensions as PyTorch extensions root...
No modifications detected for re-loaded extension module PyInit_linear_mul8s_acc, skipping build step...
Loading extension module PyInit_linear_mul8s_acc...
Using /root/.cache/torch_extensions as PyTorch extensions root...
No modifications detected for re-loaded extension module PyInit_linear_mul8s_acc, skipping build step...
Loading extension module PyInit_linear_mul8s_acc...
fc1.quantizer                           : TensorQuantizer(8bit per-tensor amax=0.9995 calibrator=HistogramCalibrator quant)
fc1.quantizer_w                         : TensorQuantizer(8bit per-tensor amax=0.3770 calibrator=HistogramCalibrator quant)
fc2.quantizer                           : TensorQuantizer(8bit per-tensor amax=2.94

W0710 19:35:12.833062 128251914852160 tensor_quantizer.py:173] Disable HistogramCalibrator
W0710 19:35:12.834116 128251914852160 tensor_quantizer.py:173] Disable HistogramCalibrator
W0710 19:35:12.835298 128251914852160 tensor_quantizer.py:173] Disable HistogramCalibrator
W0710 19:35:12.836353 128251914852160 tensor_quantizer.py:173] Disable HistogramCalibrator
W0710 19:35:12.837499 128251914852160 tensor_quantizer.py:173] Disable HistogramCalibrator
W0710 19:35:12.838432 128251914852160 tensor_quantizer.py:173] Disable HistogramCalibrator
W0710 19:35:12.840407 128251914852160 tensor_quantizer.py:237] Load calibrated amax, shape=torch.Size([]).
W0710 19:35:12.845510 128251914852160 tensor_quantizer.py:237] Load calibrated amax, shape=torch.Size([]).
W0710 19:35:12.848021 128251914852160 tensor_quantizer.py:237] Load calibrated amax, shape=torch.Size([]).
W0710 19:35:12.850494 128251914852160 tensor_quantizer.py:237] Load calibrated amax, shape=torch.Size([]).
W0710 19:35:12.852287 1282

fc1.quantizer                           : TensorQuantizer(8bit per-tensor amax=0.9995 calibrator=HistogramCalibrator quant)
fc1.quantizer_w                         : TensorQuantizer(8bit per-tensor amax=0.3770 calibrator=HistogramCalibrator quant)
fc2.quantizer                           : TensorQuantizer(8bit per-tensor amax=2.9435 calibrator=HistogramCalibrator quant)
fc2.quantizer_w                         : TensorQuantizer(8bit per-tensor amax=0.3853 calibrator=HistogramCalibrator quant)
fc3.quantizer                           : TensorQuantizer(8bit per-tensor amax=5.5282 calibrator=HistogramCalibrator quant)
fc3.quantizer_w                         : TensorQuantizer(8bit per-tensor amax=0.7954 calibrator=HistogramCalibrator quant)


In [22]:
def get_predictions(model, loader):
    model.eval()
    preds = []
    labels = []

    with torch.no_grad():
        for data, target in loader:
            data = data.cpu()
            output = model(data)
            pred = output.data.max(1)[1]

            preds.append(pred.cpu())
            labels.append(target.cpu())

    return torch.cat(preds), torch.cat(labels)

In [23]:
pred_exact, labels = get_predictions(model_exact, test_loader)
pred_approx, _ = get_predictions(model_approx, test_loader)

different = (pred_exact != pred_approx).sum().item()

print("Different predictions:", different)
print("Fraction different:", different / len(labels))

Different predictions: 689
Fraction different: 0.0689


In [24]:
def evaluate_accuracy(model, loader):
    model.eval()
    correct = 0
    total = 0

    with torch.no_grad():
        for data, target in loader:
            data = data.cpu()
            target = target.cpu()

            output = model(data)
            pred = output.data.max(1)[1]

            correct += pred.eq(target.data).cpu().sum().item()
            total += target.size(0)

    return correct / total

In [25]:
acc_exact = evaluate_accuracy(model_exact, test_loader)
acc_approx = evaluate_accuracy(model_approx, test_loader)

print("Exact accuracy:", acc_exact)
print("Approx accuracy:", acc_approx)
print("Accuracy drop:", acc_exact - acc_approx)

Exact accuracy: 0.9711
Approx accuracy: 0.9206
Accuracy drop: 0.05049999999999999


In [28]:
def get_outputs(model, loader):
    model.eval()
    outputs = []
    labels = []

    with torch.no_grad():
        for data, target in loader:
            data = data.cpu()
            output = model(data)

            outputs.append(output.cpu())
            labels.append(target.cpu())

    return torch.cat(outputs), torch.cat(labels)


out_exact, labels_exact = get_outputs(model_exact, test_loader)
out_approx, labels_approx = get_outputs(model_approx, test_loader)

print("Labels match:", torch.equal(labels_exact, labels_approx))

diff = (out_exact - out_approx).abs()

print("Mean absolute logit diff:", diff.mean().item())
print("Max absolute logit diff:", diff.max().item())
print("Number of nonzero logit differences:", (diff != 0).sum().item())

Labels match: True
Mean absolute logit diff: 4.177294731140137
Max absolute logit diff: 15.833749771118164
Number of nonzero logit differences: 100000


In [29]:
print("Exact fc1 multiplier:", model_exact.fc1.axx_mult)
print("Approx fc1 multiplier:", model_approx.fc1.axx_mult)

print("Exact fc2 multiplier:", model_exact.fc2.axx_mult)
print("Approx fc2 multiplier:", model_approx.fc2.axx_mult)

print("Exact fc3 multiplier:", model_exact.fc3.axx_mult)
print("Approx fc3 multiplier:", model_approx.fc3.axx_mult)

Exact fc1 multiplier: mul8s_acc
Approx fc1 multiplier: vakili_wrapper_generated_clean
Exact fc2 multiplier: mul8s_acc
Approx fc2 multiplier: vakili_wrapper_generated_clean
Exact fc3 multiplier: mul8s_acc
Approx fc3 multiplier: vakili_wrapper_generated_clean
